# Universal ASR Plug-and-Play Trainer

Change `MODEL_ID` and `DATASET_NAME` (or the matching environment variables) and re-run the notebook. The notebook now keeps dataset preparation, training, prediction generation, and prediction scoring as separate phases so the held-out evaluation logic stays model-agnostic.

## 0. Environment install commands

Run these in a terminal on the VM if needed:

```bash
python3.12 -m venv .venv
source .venv/bin/activate
pip install -U pip wheel setuptools
pip install -r requirements.txt
```

The smoke runner also expects `qwen-asr` if you want the Qwen prediction phase.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import yaml

CANDIDATE_PROJECT_DIRS = [
    Path.cwd(),
    Path.cwd() / "Runs" / "asr_plug_play_trainer",
    Path("/home/MohammadNabulsi/whisper/Runs/asr_plug_play_trainer"),
]
PROJECT_DIR = next(path.resolve() for path in CANDIDATE_PROJECT_DIRS if (path / "asr_universal_trainer.py").exists())
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from asr_universal_trainer import slugify

DATASET_PREP_SCRIPT = PROJECT_DIR / "prepare_smoke_datasets.py"
TRAINER_SCRIPT = PROJECT_DIR / "asr_universal_trainer.py"
PREDICT_SCRIPT = PROJECT_DIR / "generate_predictions.py"
SCORE_SCRIPT = PROJECT_DIR / "evaluate_predictions.py"
CONFIG_TEMPLATE = PROJECT_DIR / "example_config.yaml"
DATASET_REGISTRY = PROJECT_DIR / "datasets" / "registry.json"
RUNS_DIR = PROJECT_DIR / "runs"

MODEL_ID = os.getenv("ASR_PNP_MODEL_ID", "openai/whisper-large-v3")
DATASET_NAME = os.getenv("ASR_PNP_DATASET_NAME", "omnilingual_300m_levantine_full_from_whisper_splits")
RUN_NAME = os.getenv("ASR_PNP_RUN_NAME", f"{DATASET_NAME}__{MODEL_ID.replace('/', "__")}")
WORK_DIR = Path(os.getenv("ASR_PNP_WORK_DIR", str(RUNS_DIR / RUN_NAME))).resolve()
PREDICT_SPLIT = os.getenv("ASR_PNP_PREDICT_SPLIT", "test")
IS_SMOKE_DATASET = DATASET_NAME.startswith("synthetic_")
FORCE_LONG_POLICY = os.getenv("ASR_PNP_LONG_AUDIO_POLICY", "").strip()
STAGE_PLAN_ENV = os.getenv("ASR_PNP_STAGE_PLAN", "").strip()

def default_stage_plan(model_id: str) -> list[str]:
    lowered = model_id.lower()
    if any(token in lowered for token in ["whisper", "wav2vec", "hubert", "wavlm"]):
        return ["prepare", "train", "predict", "score"]
    if "qwen" in lowered or "cohere-transcribe" in lowered:
        return ["prepare", "predict", "score"]
    return ["prepare"]

STAGE_PLAN = [part.strip() for part in STAGE_PLAN_ENV.split(",") if part.strip()] or default_stage_plan(MODEL_ID)

def run_cmd(cmd: list[str]) -> subprocess.CompletedProcess[str]:
    print("Running:", " ".join(str(part) for part in cmd))
    result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:\n" + result.stderr)
    assert result.returncode == 0, result.returncode
    return result

def render_config() -> tuple[dict, Path]:
    cfg = yaml.safe_load(CONFIG_TEMPLATE.read_text(encoding="utf-8"))
    cfg["model"]["model_id"] = MODEL_ID
    cfg["model"]["gradient_checkpointing"] = False
    if "qwen" in MODEL_ID.lower():
        cfg["model"]["qwen_language"] = cfg["model"].get("qwen_language", "Arabic")
    cfg["data"] = {
        "dataset_name": DATASET_NAME,
        "dataset_registry_path": str(DATASET_REGISTRY),
        "language": cfg.get("data", {}).get("language", "ar"),
    }
    cfg["data"]["long_audio_policy"] = FORCE_LONG_POLICY or ("keep" if "qwen" in MODEL_ID.lower() else "drop")

    train_epochs_default = "1" if IS_SMOKE_DATASET else str(cfg["training"].get("num_train_epochs", 20))
    train_batch_default = "1" if IS_SMOKE_DATASET else str(cfg["training"].get("per_device_train_batch_size", 4))
    eval_batch_default = "1" if IS_SMOKE_DATASET else str(cfg["training"].get("per_device_eval_batch_size", 4))
    grad_accum_default = "1" if IS_SMOKE_DATASET else str(cfg["training"].get("gradient_accumulation_steps", 4))
    logging_steps_default = "1" if IS_SMOKE_DATASET else str(cfg["training"].get("logging_steps", 10))
    save_total_limit_default = "1" if IS_SMOKE_DATASET else str(cfg["training"].get("save_total_limit", 3))

    cfg["training"]["num_train_epochs"] = int(os.getenv("ASR_PNP_TRAIN_EPOCHS", os.getenv("ASR_PNP_SMOKE_EPOCHS", train_epochs_default)))
    cfg["training"]["per_device_train_batch_size"] = int(os.getenv("ASR_PNP_TRAIN_BATCH", os.getenv("ASR_PNP_SMOKE_BATCH", train_batch_default)))
    cfg["training"]["per_device_eval_batch_size"] = int(os.getenv("ASR_PNP_EVAL_BATCH", eval_batch_default))
    cfg["training"]["gradient_accumulation_steps"] = int(os.getenv("ASR_PNP_GRAD_ACCUM", grad_accum_default))
    cfg["training"]["logging_steps"] = int(os.getenv("ASR_PNP_LOGGING_STEPS", logging_steps_default))
    cfg["training"]["save_total_limit"] = int(os.getenv("ASR_PNP_SAVE_TOTAL_LIMIT", save_total_limit_default))
    cfg["training"]["report_to"] = os.getenv("ASR_PNP_REPORT_TO", str(cfg["training"].get("report_to", "none")))
    cfg["training"]["bf16"] = os.getenv("ASR_PNP_BF16", "false").strip().lower() in {"1", "true", "yes", "y"}
    cfg["training"]["fp16"] = os.getenv("ASR_PNP_FP16", "false").strip().lower() in {"1", "true", "yes", "y"}
    if IS_SMOKE_DATASET:
        cfg["training"]["bf16"] = False
        cfg["training"]["fp16"] = False
    cfg["evaluation"]["split"] = PREDICT_SPLIT
    cfg["output"]["work_dir"] = str(WORK_DIR)
    config_dir = RUNS_DIR / "notebook_inputs"
    config_dir.mkdir(parents=True, exist_ok=True)
    config_path = config_dir / f"{RUN_NAME}.yaml"
    config_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
    return cfg, config_path

CFG, RENDERED_CONFIG_PATH = render_config()
PREDICTION_PATH = WORK_DIR / "predictions" / f"{PREDICT_SPLIT}_{slugify(MODEL_ID)}.jsonl"
SCORE_PATH = WORK_DIR / "scores" / f"{PREDICTION_PATH.stem}_metrics.json"

print(json.dumps({
    "project_dir": str(PROJECT_DIR),
    "model_id": MODEL_ID,
    "dataset_name": DATASET_NAME,
    "stage_plan": STAGE_PLAN,
    "work_dir": str(WORK_DIR),
    "rendered_config": str(RENDERED_CONFIG_PATH),
    "prediction_path": str(PREDICTION_PATH),
    "score_path": str(SCORE_PATH),
}, ensure_ascii=False, indent=2))


## 1. Refresh the dataset registry only when needed

Synthetic smoke datasets are still available, but full runs can reuse an already materialized dataset directly from the registry without rebuilding it.


In [ ]:
registry_before = json.loads(DATASET_REGISTRY.read_text(encoding="utf-8")) if DATASET_REGISTRY.exists() else {"datasets": {}}
if DATASET_NAME not in registry_before.get("datasets", {}):
    run_cmd([sys.executable, str(DATASET_PREP_SCRIPT)])
    print(f"Registered dataset by running: {DATASET_PREP_SCRIPT.name}")
else:
    print(f"Skipping dataset preparation; reusing registered dataset: {DATASET_NAME}")
print(DATASET_REGISTRY.read_text(encoding="utf-8"))


## 2. Show the rendered config

For ordinary use, these are the only two inputs you normally change: `MODEL_ID` and `DATASET_NAME`.

In [ ]:
print(RENDERED_CONFIG_PATH.read_text(encoding="utf-8"))


## 3. Run the planned stages

The notebook intentionally keeps `predict` and `score` separate. That lets us reuse the same scoring logic regardless of whether the predictions came from Whisper, a CTC model, or Qwen.

In [ ]:
stage_results = {}
for stage in STAGE_PLAN:
    if stage in {"prepare", "train"}:
        result = run_cmd([sys.executable, str(TRAINER_SCRIPT), "--config", str(RENDERED_CONFIG_PATH), "--stage", stage])
    elif stage == "predict":
        result = run_cmd([sys.executable, str(PREDICT_SCRIPT), "--config", str(RENDERED_CONFIG_PATH), "--split", PREDICT_SPLIT])
    elif stage == "score":
        result = run_cmd([sys.executable, str(SCORE_SCRIPT), "--config", str(RENDERED_CONFIG_PATH), "--predictions", str(PREDICTION_PATH)])
    else:
        raise ValueError(f"Unsupported notebook stage: {stage}")
    stage_results[stage] = result.stdout.strip()

print(json.dumps({"completed_stages": list(stage_results)}, ensure_ascii=False, indent=2))


## 4. Inspect outputs

In [ ]:
for rel in [
    "run_result.json",
    "run.log",
    "prepared/stats.json",
    "prepared/dataset_resolved.json",
    f"predictions/{PREDICTION_PATH.name}",
    f"predictions/{PREDICTION_PATH.stem}_metadata.json",
    f"scores/{SCORE_PATH.name}",
]:
    path = WORK_DIR / rel
    print(f"\n--- {path} ---")
    if path.exists():
        print(path.read_text(encoding="utf-8")[-5000:])
    else:
        print("missing")
